# TP08: Proyecto Final
## Laboratorio (Herramientas) - Universidad del Aconcagua
### Unidad 4: Desarrollo de Proyectos Integradores

---

### 🎯 Objetivos del Trabajo Práctico

1. Resolver un **problema analítico completo**
2. Presentar un **informe ejecutivo** con dashboard
3. Defender la **documentación técnica** generada
4. Aplicar **pensamiento crítico** y soluciones basadas en datos

---

### 📁 Proyecto Final: Análisis Integral de Negocio

Proyecto final integrador que resuelve un problema real de negocio.

### 🕰️ Duración Estimada: 6 horas

In [0]:
# PROYECTO FINAL INTEGRADOR
# Optimización de Rentabilidad - Panadería La Espiga Dorada

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')

spark = SparkSession.builder.getOrCreate()

print("🎯 PROYECTO FINAL: OPTIMIZACIÓN DE RENTABILIDAD")
print("=" * 80)
print("\nPanadería La Espiga Dorada")
print("Análisis integral para mejorar la rentabilidad del negocio")
print(f"\nFecha de análisis: {pd.Timestamp.now().strftime('%Y-%m-%d %H:%M:%S')}")

## 🏪 Contexto del Negocio

### La Situación

**Panadería La Espiga Dorada** tiene 3 sucursales y ha experimentado crecimiento en ventas, pero la gerencia ha notado que **el margen de ganancia no crece proporcionalmente**.

### El Desafío

La dirección del negocio nos ha encargado:

> *"Necesitamos identificar oportunidades para mejorar la rentabilidad sin perder clientes. ¿Qué productos, sucursales y segmentos de clientes debemos optimizar?"*

### Objetivos del Proyecto

1. 📉 **Identificar productos poco rentables** que consumen recursos sin generar margen adecuado
2. 👥 **Segmentar clientes** para estrategias de marketing diferenciadas
3. 🏢 **Optimizar el mix de productos** por sucursal según rendimiento
4. 📊 **Recomendar acciones concretas** para mejorar la rentabilidad

### Métricas de Éxito

* Incrementar el margen promedio en 5 puntos porcentuales
* Reducir productos de bajo movimiento en 20%
* Aumentar la frecuencia de compra de clientes VIP en 30%
* Mejorar la eficiencia de inventario por sucursal

## 📊 Metodología de Análisis

### Enfoque del Proyecto

Aplicaremos un análisis integral que combina:

1. **Análisis de Rentabilidad por Producto**
   - Matriz de rentabilidad vs. volumen
   - Identificación de productos "estrella" y "perros"
   - Contribución marginal por categoría

2. **Segmentación de Clientes (RFM)**
   - Recency (Reciente): ºCuándo fue la última compra?
   - Frequency (Frecuencia): ¿Qué tan seguido compra?
   - Monetary (Monetario): ¿Cuánto gasta?

3. **Análisis de Eficiencia por Sucursal**
   - Rentabilidad por zona
   - Productos con mejor desempeño por sucursal
   - Oportunidades de optimización de portafolio

### Herramientas Utilizadas

* **Pandas/PySpark**: Procesamiento de datos
* **Matplotlib/Seaborn**: Visualizaciones
* **Scikit-learn**: Clustering de clientes
* **Delta Lake**: Persistencia de resultados

In [0]:
# CARGA Y PREPARACIÓN DE DATOS
print("\n📂 FASE 1: CARGA Y PREPARACIÓN DE DATOS")
print("=" * 80)

ruta_datos = '/Workspace/Users/cortega@uda.edu.ar/Laboratorio/Datasets/'

# Cargar datasets
df_productos = pd.read_csv(ruta_datos + 'productos.csv')
df_sucursales = pd.read_csv(ruta_datos + 'sucursales.csv')
df_clientes = pd.read_csv(ruta_datos + 'clientes.csv', parse_dates=['fecha_registro'])
df_ventas = pd.read_csv(ruta_datos + 'ventas.csv', parse_dates=['fecha'])
df_detalles = pd.read_csv(ruta_datos + 'detalles_ventas.csv')

# Consolidar dataset maestro
df_maestro = df_detalles.merge(df_ventas[['venta_id', 'fecha', 'sucursal_id', 'cliente_id']], on='venta_id')
df_maestro = df_maestro.merge(df_productos[['producto_id', 'nombre', 'categoria', 'costo_unitario']], on='producto_id')
df_maestro = df_maestro.merge(df_sucursales[['sucursal_id', 'zona']], on='sucursal_id')

# Calcular métricas de rentabilidad
df_maestro['costo_total'] = df_maestro['costo_unitario'] * df_maestro['cantidad']
df_maestro['ganancia'] = df_maestro['subtotal'] - df_maestro['costo_total']
df_maestro['margen_%'] = (df_maestro['ganancia'] / df_maestro['subtotal'] * 100).round(2)

print(f"\n✅ Datos consolidados: {len(df_maestro):,} registros")
print(f"\n📊 Resumen general:")
print(f"  Facturación total: ${df_maestro['subtotal'].sum():,.2f}")
print(f"  Ganancia total: ${df_maestro['ganancia'].sum():,.2f}")
print(f"  Margen promedio: {(df_maestro['ganancia'].sum() / df_maestro['subtotal'].sum() * 100):.2f}%")

## 📉 ANÁLISIS 1: Matriz de Rentabilidad por Producto

### Objetivo
Clasificar productos según su contribución a ventas y rentabilidad

### Criterios de Clasificación

* **ESTRELLA** ⭐: Alto volumen + Alto margen (mantener y promover)
* **VACA LECHERA** 🐄: Alto volumen + Bajo margen (optimizar precio/costo)
* **INTERROGANTE** ❓: Bajo volumen + Alto margen (aumentar visibilidad)
* **PERRO** 🐶: Bajo volumen + Bajo margen (descontinuar o replantear)

In [0]:
# ANÁLISIS DE RENTABILIDAD POR PRODUCTO
print("\n📊 FASE 2: ANÁLISIS DE RENTABILIDAD POR PRODUCTO")
print("=" * 80)

# Agregar por producto
analisis_productos = df_maestro.groupby(['producto_id', 'nombre', 'categoria']).agg({
    'cantidad': 'sum',
    'subtotal': 'sum',
    'ganancia': 'sum',
    'venta_id': 'nunique'
}).reset_index()

analisis_productos.columns = ['producto_id', 'nombre', 'categoria', 'unidades_vendidas', 
                               'facturacion', 'ganancia', 'num_ventas']

analisis_productos['margen_%'] = (analisis_productos['ganancia'] / analisis_productos['facturacion'] * 100).round(2)
analisis_productos['contribucion_facturacion_%'] = (analisis_productos['facturacion'] / analisis_productos['facturacion'].sum() * 100).round(2)
analisis_productos['contribucion_ganancia_%'] = (analisis_productos['ganancia'] / analisis_productos['ganancia'].sum() * 100).round(2)

print(f"\n✅ Análisis de {len(analisis_productos)} productos completado")
print("\n🔝 Top 10 productos por facturación:")
display(analisis_productos.nlargest(10, 'facturacion')[['nombre', 'categoria', 'unidades_vendidas', 'facturacion', 'margen_%']])

In [0]:
# Clasificar productos en cuadrantes
mediana_volumen = analisis_productos['facturacion'].median()
mediana_margen = analisis_productos['margen_%'].median()

def clasificar_producto(row):
    if row['facturacion'] >= mediana_volumen and row['margen_%'] >= mediana_margen:
        return '⭐ ESTRELLA'
    elif row['facturacion'] >= mediana_volumen and row['margen_%'] < mediana_margen:
        return '🐄 VACA LECHERA'
    elif row['facturacion'] < mediana_volumen and row['margen_%'] >= mediana_margen:
        return '❓ INTERROGANTE'
    else:
        return '🐶 PERRO'

analisis_productos['clasificacion'] = analisis_productos.apply(clasificar_producto, axis=1)

print("\n🎯 CLASIFICACIÓN DE PRODUCTOS")
print("=" * 80)
print("\n📊 Distribución por cuadrante:")
print(analisis_productos['clasificacion'].value_counts())

print("\n🐶 PRODUCTOS 'PERRO' (candidatos a descontinuar):")
perros = analisis_productos[analisis_productos['clasificacion'] == '🐶 PERRO'].sort_values('facturacion')
display(perros[['nombre', 'categoria', 'facturacion', 'margen_%', 'unidades_vendidas']].head(10))

In [0]:
# Visualizar matriz de portafolio
fig, ax = plt.subplots(figsize=(14, 8))

# Colores por clasificación
color_map = {
    '⭐ ESTRELLA': '#FFD700',
    '🐄 VACA LECHERA': '#4CAF50',
    '❓ INTERROGANTE': '#2196F3',
    '🐶 PERRO': '#F44336'
}

for clasificacion, color in color_map.items():
    subset = analisis_productos[analisis_productos['clasificacion'] == clasificacion]
    ax.scatter(subset['facturacion'], subset['margen_%'], 
               s=subset['unidades_vendidas']/10, alpha=0.6, 
               c=color, label=clasificacion, edgecolors='black', linewidth=0.5)

# Líneas de medianas
ax.axvline(mediana_volumen, color='gray', linestyle='--', linewidth=1, alpha=0.7, label='Mediana Facturación')
ax.axhline(mediana_margen, color='gray', linestyle='--', linewidth=1, alpha=0.7, label='Mediana Margen')

ax.set_xlabel('Facturación Total ($)', fontsize=12, fontweight='bold')
ax.set_ylabel('Margen (%)', fontsize=12, fontweight='bold')
ax.set_title('🎯 Matriz de Portafolio de Productos\n(Tamaño = Unidades Vendidas)', 
             fontsize=14, fontweight='bold', pad=20)
ax.legend(loc='best', fontsize=10)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\n✅ Matriz de portafolio generada")

## 👥 ANÁLISIS 2: Segmentación de Clientes (RFM)

### Objetivo
Segmentar clientes según comportamiento de compra para estrategias diferenciadas

### Método RFM

* **R (Recency)**: Días desde la última compra
* **F (Frequency)**: Número total de compras
* **M (Monetary)**: Gasto total acumulado

### Segmentos Esperados

* **Campeones** 🏆: R alto, F alto, M alto (clientes más valiosos)
* **Leales** 💖: F alto (compran frecuentemente)
* **Nuevos** 🌱: R alto, F bajo (recién llegados)
* **En Riesgo** ⚠️: R bajo (no compran hace tiempo)
* **Perdidos** ❌: R muy bajo, F bajo

In [0]:
# SEGMENTACIÓN DE CLIENTES RFM
print("\n👥 FASE 3: SEGMENTACIÓN DE CLIENTES (RFM)")
print("=" * 80)

# Filtrar solo ventas con cliente identificado
df_clientes_ventas = df_maestro[df_maestro['cliente_id'].notna()].copy()

# Fecha de referencia (fecha más reciente + 1 día)
fecha_referencia = df_clientes_ventas['fecha'].max() + pd.Timedelta(days=1)

# Calcular RFM por cliente
rfm = df_clientes_ventas.groupby('cliente_id').agg({
    'fecha': lambda x: (fecha_referencia - x.max()).days,  # Recency
    'venta_id': 'nunique',  # Frequency
    'subtotal': 'sum'  # Monetary
}).reset_index()

rfm.columns = ['cliente_id', 'recency', 'frequency', 'monetary']

# Join con info de clientes
rfm = rfm.merge(df_clientes[['cliente_id', 'es_vip']], on='cliente_id', how='left')

print(f"\n✅ Análisis RFM de {len(rfm)} clientes completado")
print("\n📊 Estadísticas RFM:")
print(rfm[['recency', 'frequency', 'monetary']].describe().round(2))

In [0]:
# Crear scores RFM (1-5, donde 5 es mejor)
rfm['R_score'] = pd.qcut(rfm['recency'], 5, labels=[5, 4, 3, 2, 1])  # Invertido: menor recency = mejor
rfm['F_score'] = pd.qcut(rfm['frequency'].rank(method='first'), 5, labels=[1, 2, 3, 4, 5])
rfm['M_score'] = pd.qcut(rfm['monetary'].rank(method='first'), 5, labels=[1, 2, 3, 4, 5])

# Convertir a int
rfm['R_score'] = rfm['R_score'].astype(int)
rfm['F_score'] = rfm['F_score'].astype(int)
rfm['M_score'] = rfm['M_score'].astype(int)

# Score RFM combinado
rfm['RFM_score'] = rfm['R_score'].astype(str) + rfm['F_score'].astype(str) + rfm['M_score'].astype(str)
rfm['RFM_score_num'] = rfm['R_score'] + rfm['F_score'] + rfm['M_score']

# Segmentar clientes
def segmentar_cliente(row):
    if row['RFM_score_num'] >= 13:
        return '🏆 Campeones'
    elif row['F_score'] >= 4:
        return '💖 Leales'
    elif row['R_score'] >= 4 and row['F_score'] <= 2:
        return '🌱 Nuevos'
    elif row['R_score'] <= 2 and row['F_score'] >= 3:
        return '⚠️ En Riesgo'
    elif row['R_score'] <= 2:
        return '❌ Perdidos'
    else:
        return '👤 Regulares'

rfm['segmento'] = rfm.apply(segmentar_cliente, axis=1)

print("\n🎯 SEGMENTACIÓN DE CLIENTES")
print("=" * 80)
print("\n📊 Distribución por segmento:")
segmentacion = rfm.groupby('segmento').agg({
    'cliente_id': 'count',
    'monetary': 'sum',
    'frequency': 'mean'
}).round(2)
segmentacion.columns = ['Num_Clientes', 'Valor_Total', 'Frecuencia_Promedio']
segmentacion['Valor_%'] = (segmentacion['Valor_Total'] / segmentacion['Valor_Total'].sum() * 100).round(2)
display(segmentacion.sort_values('Valor_Total', ascending=False))

In [0]:
# Visualizar segmentos
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# Gráfico 1: Número de clientes por segmento
segmento_counts = rfm['segmento'].value_counts()
colors = ['#FFD700', '#4CAF50', '#8BC34A', '#FFC107', '#FF5722', '#9E9E9E']
ax1.bar(range(len(segmento_counts)), segmento_counts.values, color=colors, edgecolor='black')
ax1.set_xticks(range(len(segmento_counts)))
ax1.set_xticklabels(segmento_counts.index, rotation=45, ha='right')
ax1.set_ylabel('Número de Clientes', fontsize=12, fontweight='bold')
ax1.set_title('👥 Distribución de Clientes por Segmento', fontsize=14, fontweight='bold')
ax1.grid(axis='y', alpha=0.3)
for i, v in enumerate(segmento_counts.values):
    ax1.text(i, v, str(v), ha='center', va='bottom', fontsize=10, fontweight='bold')

# Gráfico 2: Valor total por segmento
segmento_valor = rfm.groupby('segmento')['monetary'].sum().sort_values(ascending=False)
ax2.barh(range(len(segmento_valor)), segmento_valor.values, color=colors, edgecolor='black')
ax2.set_yticks(range(len(segmento_valor)))
ax2.set_yticklabels(segmento_valor.index)
ax2.set_xlabel('Valor Total ($)', fontsize=12, fontweight='bold')
ax2.set_title('💰 Valor Total Generado por Segmento', fontsize=14, fontweight='bold')
ax2.grid(axis='x', alpha=0.3)
for i, v in enumerate(segmento_valor.values):
    ax2.text(v, i, f'${v:,.0f}', ha='left', va='center', fontsize=9, fontweight='bold')

plt.tight_layout()
plt.show()

print("\n✅ Segmentación visual completada")

## 🏢 ANÁLISIS 3: Optimización por Sucursal

### Objetivo
Identificar oportunidades específicas de optimización en cada sucursal

### Preguntas Clave

1. ¿Qué productos tienen mejor desempeño en cada zona?
2. ¿Dónde están los mayores márgenes de mejora?
3. ¿Cómo optimizar el mix de productos por sucursal?

In [0]:
# ANÁLISIS POR SUCURSAL
print("\n🏢 FASE 4: OPTIMIZACIÓN POR SUCURSAL")
print("=" * 80)

# KPIs por sucursal
kpi_sucursales = df_maestro.groupby('zona').agg({
    'venta_id': 'nunique',
    'subtotal': 'sum',
    'ganancia': 'sum',
    'cantidad': 'sum'
}).reset_index()

kpi_sucursales['margen_%'] = (kpi_sucursales['ganancia'] / kpi_sucursales['subtotal'] * 100).round(2)
kpi_sucursales['ticket_promedio'] = (kpi_sucursales['subtotal'] / kpi_sucursales['venta_id']).round(2)

kpi_sucursales.columns = ['Zona', 'Num_Ventas', 'Facturacion', 'Ganancia', 'Unidades', 'Margen_%', 'Ticket_Promedio']

print("\n📊 KPIs POR SUCURSAL")
print("=" * 80)
display(kpi_sucursales)

print("\n🔍 Insights:")
print(f"  Mejor margen: {kpi_sucursales.loc[kpi_sucursales['Margen_%'].idxmax(), 'Zona']} ({kpi_sucursales['Margen_%'].max():.2f}%)")
print(f"  Mayor facturación: {kpi_sucursales.loc[kpi_sucursales['Facturacion'].idxmax(), 'Zona']} (${kpi_sucursales['Facturacion'].max():,.2f})")
print(f"  Mayor ticket: {kpi_sucursales.loc[kpi_sucursales['Ticket_Promedio'].idxmax(), 'Zona']} (${kpi_sucursales['Ticket_Promedio'].max():,.2f})")

In [0]:
# Top productos por zona
print("\n🏆 TOP 5 PRODUCTOS POR ZONA")
print("=" * 80)

for zona in df_maestro['zona'].unique():
    print(f"\n📍 {zona}:")
    top_zona = df_maestro[df_maestro['zona'] == zona].groupby('nombre').agg({
        'subtotal': 'sum',
        'ganancia': 'sum'
    }).sort_values('subtotal', ascending=False).head(5)
    top_zona['margen_%'] = (top_zona['ganancia'] / top_zona['subtotal'] * 100).round(2)
    display(top_zona)

## 💡 RECOMENDACIONES ESTRATÉGICAS

### Basadas en los hallazgos del análisis

---

### 1️⃣ OPTIMIZACIÓN DE PORTAFOLIO DE PRODUCTOS

#### Acciones Inmediatas:

* **Productos PERRO 🐶** (Bajo volumen + Bajo margen):
  - **Descontinuar** productos con menos de 100 unidades vendidas en 2 años
  - **Replantear** precio o promoción de productos con potencial
  - **Liberar** espacio de exhibición para productos más rentables
  - **Impacto estimado**: Aumento de 3-5% en margen general

* **Productos ESTRELLA ⭐** (Alto volumen + Alto margen):
  - **Aumentar** visibilidad en puntos de venta
  - **Garantizar** disponibilidad constante (evitar quiebres de stock)
  - **Promover** en campañas de marketing
  - **Considerar** extensión de línea (variantes, tamaños)

* **Productos INTERROGANTE ❓** (Bajo volumen + Alto margen):
  - **Probar** campañas de degustación
  - **Mejorar** ubicación en tienda
  - **Educar** al cliente sobre valor diferencial
  - **Potencial**: Convertir en ESTRELLA

---

### 2️⃣ ESTRATEGIAS POR SEGMENTO DE CLIENTES

#### **Campeones 🏆** (Alto valor, alta frecuencia):
* **Programa de fidelización premium** con beneficios exclusivos
* **Comunicación personalizada** sobre nuevos productos
* **Eventos especiales** (catas, lanzamientos)
* **Objetivo**: Mantener y aumentar gasto promedio

#### **Leales 💖** (Compran frecuentemente):
* **Sistema de puntos** acumulables
* **Descuentos progresivos** por volumen
* **Early access** a promociones
* **Objetivo**: Convertir en Campeones

#### **En Riesgo ⚠️** (No compran hace tiempo):
* **Campaña de reactivación** con cupón de descuento
* **Encuesta** para entender motivo de alejamiento
* **Oferta especial** "Te extrañamos"
* **Objetivo**: Recuperar el 30% en 3 meses

#### **Nuevos 🌱** (Recién llegados):
* **Welcome pack** con productos variados
* **Educación** sobre la propuesta de valor
* **Seguimiento** en primeras 3 compras
* **Objetivo**: Convertir en Leales

---

### 3️⃣ OPTIMIZACIÓN POR SUCURSAL

#### **Centro** (Margen alto, ticket alto):
* **Estrategia**: Premium, productos gourmet
* **Foco**: Tortas, productos especiales
* **Horarios extendidos** para captar tráfico de oficinas

#### **Este** (Volumen alto):
* **Estrategia**: Volumen, productos familiares
* **Foco**: Panes, facturas surtidas
* **Promociones por cantidad** (docenas, combos)

#### **Sur**:
* **Estrategia**: Balanceada
* **Foco**: Mix de productos
* **Analizar** apertura de cafetería (como Centro y Este)

---

### 4️⃣ MÉTRICAS DE SEGUIMIENTO

#### KPIs a monitorear mensualmente:

1. **Margen promedio general** (objetivo: +5pp)
2. **Rotación de inventario** por categoría
3. **Tasa de recompra** de clientes nuevos (objetivo: 60%)
4. **Valor de vida del cliente (LTV)** por segmento
5. **Ticket promedio** por sucursal
6. **% de quiebre de stock** en productos ESTRELLA (objetivo: <2%)

---

### 📅 ROADMAP DE IMPLEMENTACIÓN

**Mes 1:**
* Descontinuar 10 productos PERRO de menor rotación
* Lanzar programa de fidelización básico
* Optimizar exhibición de productos ESTRELLA

**Mes 2:**
* Campaña de reactivación clientes En Riesgo
* Test de promociones para productos INTERROGANTE
* Análisis de costos para mejorar márgenes VACA LECHERA

**Mes 3:**
* Evaluar resultados iniciales
* Ajustar estrategias según performance
* Planificar siguiente fase de optimización

## 📊 DASHBOARD EJECUTIVO

### Resumen visual de hallazgos clave

In [0]:
# DASHBOARD EJECUTIVO FINAL
print("\n📊 DASHBOARD EJECUTIVO FINAL")
print("=" * 80)

fig = plt.figure(figsize=(18, 12))
gs = fig.add_gridspec(3, 3, hspace=0.3, wspace=0.3)

# 1. KPIs Generales (Top)
ax1 = fig.add_subplot(gs[0, :])
ax1.axis('off')
kpis_text = f"""
FACTURACIlabelIÓN TOTAL: ${df_maestro['subtotal'].sum():,.0f}  |  GANANCIA TOTAL: ${df_maestro['ganancia'].sum():,.0f}  |  MARGEN PROMEDIO: {(df_maestro['ganancia'].sum() / df_maestro['subtotal'].sum() * 100):.1f}%
VENTAS TOTALES: {df_maestro['venta_id'].nunique():,}  |  PRODUCTOS ACTIVOS: {df_maestro['producto_id'].nunique()}  |  CLIENTES ACTIVOS: {df_maestro['cliente_id'].nunique():,}
"""
ax1.text(0.5, 0.5, kpis_text, ha='center', va='center', fontsize=14, fontweight='bold',
         bbox=dict(boxstyle='round', facecolor='lightblue', alpha=0.3))

# 2. Facturación por Sucursal
ax2 = fig.add_subplot(gs[1, 0])
kpi_sucursales.plot(x='Zona', y='Facturacion', kind='bar', ax=ax2, color=['#1f77b4', '#ff7f0e', '#2ca02c'], legend=False)
ax2.set_title('🏪 Facturación por Zona', fontsize=12, fontweight='bold')
ax2.set_ylabel('Facturación ($)')
ax2.set_xlabel('')
ax2.grid(axis='y', alpha=0.3)

# 3. Margen por Sucursal
ax3 = fig.add_subplot(gs[1, 1])
kpi_sucursales.plot(x='Zona', y='Margen_%', kind='bar', ax=ax3, color='coral', legend=False)
ax3.set_title('💰 Margen por Zona', fontsize=12, fontweight='bold')
ax3.set_ylabel('Margen (%)')
ax3.set_xlabel('')
ax3.grid(axis='y', alpha=0.3)

# 4. Distribución de Productos
ax4 = fig.add_subplot(gs[1, 2])
clasif_counts = analisis_productos['clasificacion'].value_counts()
colors_clasif = ['#FFD700', '#4CAF50', '#2196F3', '#F44336']
ax4.pie(clasif_counts, labels=clasif_counts.index, autopct='%1.0f%%', colors=colors_clasif, startangle=90)
ax4.set_title('📊 Clasificación de Productos', fontsize=12, fontweight='bold')

# 5. Top 8 Productos
ax5 = fig.add_subplot(gs[2, 0])
top_8 = analisis_productos.nlargest(8, 'facturacion')
ax5.barh(range(len(top_8)), top_8['facturacion'], color='steelblue')
ax5.set_yticks(range(len(top_8)))
ax5.set_yticklabels(top_8['nombre'], fontsize=9)
ax5.set_xlabel('Facturación ($)')
ax5.set_title('🏆 Top 8 Productos', fontsize=12, fontweight='bold')
ax5.grid(axis='x', alpha=0.3)

# 6. Segmentos de Clientes
ax6 = fig.add_subplot(gs[2, 1])
seg_counts = rfm['segmento'].value_counts().head(6)
colors_seg = ['#FFD700', '#4CAF50', '#8BC34A', '#FFC107', '#FF5722', '#9E9E9E']
ax6.bar(range(len(seg_counts)), seg_counts.values, color=colors_seg)
ax6.set_xticks(range(len(seg_counts)))
ax6.set_xticklabels(seg_counts.index, rotation=45, ha='right', fontsize=9)
ax6.set_ylabel('Número de Clientes')
ax6.set_title('👥 Segmentos de Clientes', fontsize=12, fontweight='bold')
ax6.grid(axis='y', alpha=0.3)

# 7. Categorías
ax7 = fig.add_subplot(gs[2, 2])
cat_ventas = df_maestro.groupby('categoria')['subtotal'].sum().sort_values(ascending=True)
ax7.barh(range(len(cat_ventas)), cat_ventas.values, color='teal')
ax7.set_yticks(range(len(cat_ventas)))
ax7.set_yticklabels(cat_ventas.index, fontsize=9)
ax7.set_xlabel('Facturación ($)')
ax7.set_title('🍰 Ventas por Categoría', fontsize=12, fontweight='bold')
ax7.grid(axis='x', alpha=0.3)

plt.suptitle('🎯 DASHBOARD EJECUTIVO - PANADERÍA LA ESPIGA DORADA', 
             fontsize=18, fontweight='bold', y=0.98)
plt.show()

print("\n✅ Dashboard ejecutivo generado")

## ✅ CONCLUSIONES Y SIGUIENTES PASOS

---

### 🎯 Hallazgos Clave del Proyecto

#### 1. **Oportunidades de Optimización de Portafolio**

* Identificamos productos con **bajo volumen y bajo margen** que pueden descontinuarse
* Detectamos productos **ESTRELLA** que merecen mayor inversión en marketing
* Encontramos productos **INTERROGANTE** con alto potencial sin explotar
* **Impacto esperado**: Mejora de 3-5% en margen general

#### 2. **Segmentación Efectiva de Clientes**

* **Campeones** representan el mayor valor pero son minoría
* Clientes **En Riesgo** y **Perdidos** ofrecen oportunidad de recuperación
* **Nuevos** clientes necesitan programa de activación para convertirse en Leales
* **Estrategia diferenciada** por segmento puede aumentar LTV en 25-30%

#### 3. **Diferencias por Sucursal**

* **Centro**: Mejor margen, perfil premium
* **Este**: Mayor volumen, orientado a familias
* **Sur**: Potencial de mejora con ajustes de portafolio
* **Cada zona necesita estrategia diferenciada**

---

### 📊 Impacto Estimado de las Recomendaciones

| Iniciativa | Impacto Esperado | Plazo |
|---|---|---|
| Descontinuar productos PERRO | +3-5% margen | 1-2 meses |
| Programa de fidelización | +15% frecuencia compra | 3-6 meses |
| Reactivación clientes en riesgo | +10% facturación | 2-3 meses |
| Optimización mix por sucursal | +8% eficiencia inventario | 3-4 meses |
| **TOTAL ESTIMADO** | **+12-18% rentabilidad** | **6 meses** |

---

### 🚀 Siguientes Pasos Inmediatos

#### **Corto Plazo (1-2 semanas)**
1. ✅ Presentar findings a gerencia
2. ✅ Validar lista de productos a descontinuar
3. ✅ Diseñar programa de fidelización básico
4. ✅ Preparar campaña de reactivación

#### **Mediano Plazo (1-3 meses)**
1. 🔄 Implementar cambios en portafolio
2. 🔄 Lanzar programas por segmento de clientes
3. 🔄 Ajustar exhibición por sucursal
4. 🔄 Establecer sistema de monitoreo de KPIs

#### **Largo Plazo (3-6 meses)**
1. 📅 Evaluar resultados y ROI de iniciativas
2. 📅 Iterar y optimizar estrategias
3. 📅 Expandir mejores prácticas a toda la red
4. 📅 Preparar siguiente fase de transformación

---

### 📚 Aprendizajes del Proyecto

#### **Técnicos:**
* Integración de múltiples fuentes de datos
* Análisis multidimensional (producto, cliente, sucursal)
* Visualización efectiva para storytelling
* Metodologías avanzadas (RFM, portfolio matrix)

#### **De Negocio:**
* Importancia de datos para decisiones estratégicas
* Enfoque en rentabilidad vs. solo volumen
* Valor de segmentación de clientes
* Necesidad de estrategias diferenciadas por canal

---

### 🎓 Reflexión Final

Este proyecto integrador demostró cómo el **análisis de datos** puede generar **valor tangible** para un negocio:

* ✅ **Problemática identificada**: Crecimiento en ventas sin proporcional mejora en rentabilidad
* ✅ **Análisis riguroso**: Metodologías probadas aplicadas a datos reales
* ✅ **Insights accionables**: Recomendaciones concretas con impacto medible
* ✅ **Roadmap claro**: Plan de implementación priorizado

**El ciclo completo**: desde datos crudos hasta recomendaciones estratégicas.

Este es el **poder del Data Analytics** en acción. 🚀

---

## 🎆 ¡FELICITACIONES!

**Has completado el Proyecto Final del Laboratorio de Herramientas en la Nube**

### 🎓 Habilidades Demostradas:

✅ Carga y consolidación de datos desde múltiples fuentes  
✅ Limpieza y transformación de datasets complejos  
✅ Análisis exploratorio y estadístico avanzado  
✅ Visualización de datos para comunicación ejecutiva  
✅ Modelado y segmentación de clientes (RFM)  
✅ Análisis de portafolio de productos (BCG Matrix)  
✅ Pensamiento crítico y resolución de problemas  
✅ Traducción de insights a recomendaciones de negocio  
✅ Documentación técnica y ejecutiva  
✅ Presentación de resultados con storytelling  

---

### 📚 CURSO COMPLETO - 8 TRABAJOS PRÁCTICOS

**Unidad 1 - Análisis de Datos**
* ✅ TP01: Configuración Cloud y Almacenamiento
* ✅ TP02: Manipulación Programática y Exploración

**Unidad 2 - Visualización de Datos**
* ✅ TP03: Perfilado de Datos
* ✅ TP04: Dashboards Interactivos

**Unidad 3 - Modelado de Datos**
* ✅ TP05: Estructuración y Agregación
* ✅ TP06: Feature Engineering y Modelado

**Unidad 4 - Proyectos Integradores**
* ✅ TP07: Pipeline Integrador
* ✅ TP08: Proyecto Final

---

## 🌟 ¡CURSO FINALIZADO CON ÉXITO!

**Estás listo para aplicar estas herramientas en proyectos reales de Data Science y Analytics.**

---

**Universidad del Aconcagua**  
**Laboratorio (Herramientas) - Data Science**  
**Mendoza, Argentina**